# Mesh 2 · Walking a mesh

**Fundamentals, part 2 of 4.** Companion to [Week 05 §4–6](../README.md).
Do [`01_mesh_from_scratch`](01_mesh_from_scratch.ipynb) first.

A mesh earns its keep when you ask it *what is next to what*. Every such
question is answered the same way underneath: by stepping along
**halfedges**. This notebook takes those steps by hand once, so that the
convenient methods (`vertex_neighbors`, `face_neighbors`, `edge_loop` …)
stop being magic.

| You will | Section |
| --- | --- |
| read the halfedge table | 1 |
| walk round a face, then across an edge | 2–3 |
| walk round a vertex | 4 |
| find the boundary, and its trap | 5 |
| follow a loop and a strip across the whole mesh | 6 |
| grow a neighbourhood ring by ring | 7 |

In [ ]:
# Setup: run this cell first. It finds the shared drawing helper (viz.py),
# whether this notebook sits in its lecture folder or is your copy in MyWork/.
import sys
import pathlib

HERE = pathlib.Path.cwd()
NB = next(p / "Lecture" / "notebooks" for p in [HERE, *HERE.parents]
          if (p / "Lecture" / "notebooks" / "viz.py").exists())
sys.path.insert(0, str(NB))
from viz import chart, draw_graph, draw_mesh, draw_plan, draw_points, ramp, row, view3d

from compas.datastructures import Mesh


def grid(nx, ny, spacing=1.0):
    """The grid from notebook 01: keys along x first, every face counter-clockwise."""
    mesh = Mesh()
    key = {}
    for j in range(ny + 1):
        for i in range(nx + 1):
            key[i, j] = mesh.add_vertex(x=i * spacing, y=j * spacing, z=0)
    for j in range(ny):
        for i in range(nx):
            mesh.add_face([key[i, j], key[i + 1, j], key[i + 1, j + 1], key[i, j + 1]])
    return mesh

## 1 · Every edge is stored twice

![Two quads; every edge drawn as two opposite arrows, coloured by the face they belong to](../figures/mesh-halfedges.svg)

An edge between vertices 1 and 4 is kept as two **halfedges**: `(1, 4)` and
`(4, 1)`. Each remembers the face on its **left**. On the outside there is
no face, and the entry is `None`.

In [ ]:
two = grid(2, 1)
for u in two.halfedge:
    print(u, "->", dict(two.halfedge[u]))

Read one row: `1 -> {2: None, 4: 0, 0: None}` says *from vertex 1, the
halfedge to 4 has face 0 on its left; the ones to 2 and to 0 have nothing.*
The whole mesh is this dictionary of dictionaries. Everything below is a
way of reading it.

In [ ]:
print("halfedge (1, 4) belongs to face", two.halfedge_face((1, 4)))
print("halfedge (4, 1) belongs to face", two.halfedge_face((4, 1)))
print("halfedge (0, 1) belongs to face", two.halfedge_face((0, 1)))
print("halfedge (1, 0) belongs to face", two.halfedge_face((1, 0)), " <- the outside")

## 2 · Round a face

![Walking round face 0 by halfedges, and round vertex 4 by its ordered neighbours](../figures/mesh-walk.svg)

Stand on a halfedge and ask for the one **after** it in the same face.
Keep asking. You come back to where you started, having visited every
side of the face once.

In [ ]:
g = grid(2, 2)
start = (0, 1)
here = start
for step in range(1, 10):
    print(f"step {step}: halfedge {here}, face {g.halfedge_face(here)}")
    here = g.halfedge_after(here)
    if here == start:
        break
print("the same thing in one call:", g.face_halfedges(0))

## 3 · Across an edge

The twin of `(1, 4)` is `(4, 1)`. Same edge, other side, **other face**.
Flipping a halfedge is how you step from a face to its neighbour.

In [ ]:
for u, v in g.face_halfedges(0):
    print(f"across edge ({u}, {v}) from face 0 lies face", g.halfedge_face((v, u)))
print("the same thing in one call: face_neighbors(0) =", g.face_neighbors(0))

Two of the four answers are `None`: face 0 is in a corner, and two of its
sides are on the boundary. `face_neighbors` leaves those out.

## 4 · Round a vertex

![Three panels on a 4 by 4 grid: the neighbours of vertex 12, the faces around vertex 12, the neighbours of face 5](../figures/mesh-neighbours.svg)

The outgoing halfedges of a vertex lead to its neighbours. Ask for them
`ordered=True` and they come in rotation, like spokes: which makes them
usable as a polygon.

In [ ]:
v = 4
print("neighbours, in rotation:", g.vertex_neighbors(v, ordered=True))
print("faces, in rotation:     ", g.vertex_faces(v, ordered=True))
print("degree (how many edges):", g.vertex_degree(v))
row(draw_mesh(g, "vertex_neighbors(4)", highlight_vertices=g.vertex_neighbors(v),
              highlight_edges=[(v, n) for n in g.vertex_neighbors(v)], size=(400, 340)),
    draw_mesh(g, "vertex_faces(4)", highlight_faces=g.vertex_faces(v), face_labels=True, size=(400, 340)))

Degree is worth a habit. On a quad grid every inner vertex has degree 4;
on a triangle mesh, 6. A vertex with a different degree is where the
pattern has a **singularity**, and on a built gridshell that is a special
node somebody has to design.

In [ ]:
big = grid(4, 3)
for degree in sorted({big.vertex_degree(k) for k in big.vertices()}):
    keys = [k for k in big.vertices() if big.vertex_degree(k) == degree]
    print(f"degree {degree}: {len(keys):2d} vertices  {keys}")

Degree 2 is a corner, 3 an edge of the sheet, 4 the inside. You just
classified every vertex of the mesh without looking at one coordinate.

## 5 · The boundary

![A 4 by 3 grid with its boundary vertices and edges highlighted](../figures/mesh-boundary.svg)

The boundary is every halfedge whose face is `None`. It is where a roof
gets its edge beam, a fabric its hem, a shell its supports.

In [ ]:
edges = big.edges_on_boundary()
loop = big.vertices_on_boundary()
print(len(edges), "boundary edges")
print("is vertex 6 on the boundary?", big.is_vertex_on_boundary(6), "| vertex 4?", big.is_vertex_on_boundary(4))
print("the loop:", loop)

Count that loop before you use it.

In [ ]:
print("len(loop) =", len(loop), "| different vertices =", len(set(loop)))
print("first and last:", loop[0], loop[-1])
draw_mesh(big, "the boundary", highlight_vertices=set(loop), highlight_edges=edges)

**The trap.** `vertices_on_boundary()` returns a *closed walk*: the first
vertex appears again at the end. Use it as "the set of boundary vertices"
and you count one twice; pin them as supports and one gets pinned twice.
Use `set(...)` when you mean a set, and keep the list when you mean the
path (it is ready to draw as a closed polyline).

## 6 · Loops and strips: straight lines through a quad mesh

![An edge loop runs straight through vertices; an edge strip runs across a row of quads](../figures/mesh-loops-strips.svg)

On a quad mesh, "keep going straight" means something. From one edge:

- its **loop** continues end to end through each vertex: a continuous
  line, a rail, a rib
- its **strip** steps sideways across each quad to the opposite edge: a
  row of panels, a course, a plank

In [ ]:
edge = (6, 7)
loop_edges = big.edge_loop(edge)
strip_edges = big.edge_strip(edge)
print("loop: ", loop_edges)
print("strip:", strip_edges)
row(draw_mesh(big, f"edge_loop({edge})", highlight_edges=loop_edges, size=(420, 340)),
    draw_mesh(big, f"edge_strip({edge})", highlight_edges=strip_edges, size=(420, 340)))

The faces **between** the rungs of a strip are one row of panels. Collect
them with the halfedge step from section 3:

In [ ]:
row_of_panels = [big.halfedge_face(e) for e in strip_edges if big.halfedge_face(e) is not None]
print("panels in that strip:", row_of_panels)
draw_mesh(big, "one strip = one row of panels", highlight_faces=row_of_panels,
          highlight_edges=strip_edges, face_labels=True, vertex_labels=False)

A facade schedule by rows, a timber gridshell laid lath by lath, a brick
vault built course by course: each is a loop or a strip. None of them is a
coordinate query, so each still works after the mesh is bent.

## 7 · Ring by ring

Neighbours of neighbours. `vertex_neighborhood(key, ring=n)` is everything
within `n` edges: the zone a point load disturbs, the panels a column
carries, the area a fixing affects.

In [ ]:
wide = grid(8, 6)
centre = 31
rings = {0: {centre}}
for n in (1, 2, 3):
    rings[n] = set(wide.vertex_neighborhood(centre, ring=n)) - set().union(*rings.values())
    print(f"ring {n}: {len(rings[n]):2d} vertices")
shade = {0: "#e8590c", 1: "#1c7ed6", 2: "#74c0fc", 3: "#d0ebff"}
near = {k: n for n, keys in rings.items() for k in keys}
row(*[draw_mesh(wide, f"within {n} edges of vertex {centre}", vertex_labels=False, size=(300, 250),
                highlight_vertices=[k for k, r in near.items() if r <= n]) for n in (1, 2, 3)])

The rings are diamonds, not circles: distance here is counted in **edges**,
not metres. Week 06 gives that way of measuring a name (breadth-first
search) and Week 10 turns it into real distances over a surface.

## What to keep

- A halfedge knows the face on its left; its twin knows the other side.
- *Next in the face* walks round a face. *Twin* crosses an edge. Those two
  moves reach everything.
- `None` is the outside. The boundary is where you meet it.
- `vertices_on_boundary()` repeats its first vertex. Count before you use.
- On quads, loops and strips are the mesh's own straight lines.

Next: [`03_data_on_a_mesh`](03_data_on_a_mesh.ipynb).

## Your turn
1. Walk the **boundary** by hand: start at halfedge `(1, 0)` of `grid(2, 2)`
   and keep taking `halfedge_after`. Which way round does the outside go?
2. For `grid(4, 3)`, list every face with fewer than four neighbours using
   `face_neighbors`. Check against `is_face_on_boundary`.
3. Split the faces of `grid(6, 4)` into rows using `edge_strip`, one row
   per edge on the left boundary. Colour alternate rows.
4. Triangulate `grid(4, 3)` (`quads_to_triangles()` on a copy) and list the
   degrees again. Which vertices changed, and what happened to `edge_loop`?